In [1]:
import sys
import pandas as pd 
print(sys.executable)
print(pd.__version__)

D:\Dev_Env\Dev_Env_Tools\anaconda3\envs\opt_ai\python.exe
2.3.3


In [2]:
load  = pd.read_csv("../data/raw/Load_history.csv", thousands = ",")
hour_cols = [f"h{i}" for i in range(1,25)]
print(load.shape)
load[hour_cols].dtypes.unique()

(33000, 28)


array([dtype('float64')], dtype=object)

In [3]:
na = load[hour_cols].isna()
any_na = na.any(axis=1)
all_na = na.all(axis=1)
print(any_na.sum(),all_na.sum())

partial = load[any_na & ~all_na]
partial[["zone_id", "year", "month", "day"]].drop_duplicates(["year", "month", "day"])

1280 1260


,zone_id,year,month,day
1642,1,2008,6,30


In [4]:
missing = load[all_na].copy()
missing["date"] = pd.to_datetime(missing[["year", "month", "day"]])
missing_dates = missing["date"].drop_duplicates().sort_values()
print(len(missing_dates))

63


* 8 tuần backcast và 1 tuần dự báo (01/0/7/2018 - 07/07/2018) bị xóa; giá trị thật sẽ nằm trong file Load_solution.csv
* Ngày 30/06/2018: chỉ có h1-h6; 18 giờ còn lại bị mất hẳn và không có trong file Solution 

In [5]:
sol = pd.read_csv("../data/raw/Load_solution.csv", thousands = ",")
print(sol.shape)
print(sol.columns)

(1323, 30)
Index(['id', 'zone_id', 'year', 'month', 'day', 'h1', 'h2', 'h3', 'h4', 'h5',
       'h6', 'h7', 'h8', 'h9', 'h10', 'h11', 'h12', 'h13', 'h14', 'h15', 'h16',
       'h17', 'h18', 'h19', 'h20', 'h21', 'h22', 'h23', 'h24', 'weight'],
      dtype='object')


In [6]:
print(sol["zone_id"].nunique())

21


In [7]:
# Tổng 20 zone, theo từng ngày
zones_sum = (sol[sol["zone_id"] <= 20]
             .groupby(["year", "month", "day"])[hour_cols]
             .sum())

# Zone 21, cùng cách đánh chỉ mục
system = (sol[sol["zone_id"] == 21]
          .set_index(["year", "month", "day"])[hour_cols])

# Chênh lệch lớn nhất
(zones_sum - system).abs().max().max()

np.int64(0)

In [8]:
sol["is_system"] = sol["zone_id"] == 21
sol["is_forecast_week"] = sol["year"] == 2008
pd.crosstab([sol["is_system"], sol["is_forecast_week"]], sol["weight"])

weight                       1    8    20   160
is_system is_forecast_week                     
False     False             1120    0    0    0
          True                 0  140    0    0
True      False                0    0   56    0
          True                 0    0    0    7

In [9]:
sol["id"].is_unique, sol["id"].min(), sol["id"].max()

(True, np.int64(1), np.int64(1323))

## Khám phá `Load_solution.csv`

**Cấu trúc:** 1323 dòng × 30 cột = 63 ngày bị xóa × 21 zone. Có 28 cột giống `Load_history`, thêm `id` và `weight`.

| # | Câu hỏi | Kết luận | Xử lý |
|---|---|---|---|
| 1 | File chứa gì? | Giá trị thật của 63 ngày bị xóa (8 tuần backcast + 1 tuần dự báo), cho 21 zone | Dùng để lấp chỗ trống trong `Load_history` |
| 2 | Zone 21 là gì? | Tổng phụ tải toàn hệ thống, bằng chính xác tổng zone 1–20 (chênh lệch lớn nhất = 0) | Bỏ khi ghép; tự tính lại cho toàn bộ chuỗi |
| 3 | `weight` là gì? | Trọng số chấm điểm cuộc thi: zone thường 1 / 8, hệ thống 20 / 160 (backcast / dự báo) | Bỏ, vì dùng giao thức đánh giá riêng |
| 4 | `id` là gì? | Số thứ tự dòng (1–1323, không lặp) | Bỏ |

**Lưu ý:**
- Ngày 30/06/2008 không có trong file solution, nên 18 giờ (`h7`–`h24`) của ngày này vẫn bị thiếu.
- Zone 3 và zone 7 có giá trị trùng nhau ở ít nhất một giờ, cần kiểm tra khi làm EDA.

In [10]:
keys = ["zone_id", "year", "month", "day"]

sol_clean = (sol[sol["zone_id"] <= 20]
             .drop(columns=["id", "weight", "is_system", "is_forecast_week"]))
print(sol_clean.shape)

(1260, 28)


In [11]:
load_idx = load.set_index(keys)
sol_idx = sol_clean.set_index(keys)

full = load_idx.fillna(sol_idx)

In [12]:
print(full.shape)
print(full.isna().any(axis=1).sum(), full.isna().all(axis=1).sum())

(33000, 24)
20 0


In [13]:
# (a) Các ô vừa điền có đúng bằng giá trị trong sol không?
print((full.loc[sol_idx.index] - sol_idx).abs().max().max())

# (b) Các ô vốn đã có giá trị có bị thay đổi không?
mask = load_idx.notna()
print((full - load_idx).abs()[mask].max().max())

0.0
0.0
